# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library, which allows loading, examining, and processing data described using the [Croissant](https://mlcommons.github.io/croissant/) format.

### Dataset Source
The dataset is described with a Croissant schema and accessible via the following JSON-LD URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset schema and metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata (schema)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n")
print("Dataset description:\n", metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id` references.

We use `dataset.record_sets` to list all available record sets and their IDs. Then, we list the fields for a selected record set.

In [ ]:
# List all record sets and their @ids
print("Available Record Sets:")
for rset in dataset.record_sets:
    print(f"  recordSet @id: {rset.id} - name: {rset.name}")

# For this dataset, there is usually one main record set. Let's select it for further exploration.
record_set_ids = [rset.id for rset in dataset.record_sets]
main_record_set_id = record_set_ids[0] if record_set_ids else None

if main_record_set_id:
    main_rs = [rset for rset in dataset.record_sets if rset.id == main_record_set_id][0]
    print(f"\nFields in recordSet (id={main_record_set_id}):")
    for field in main_rs.fields:
        col_id = field.columns[0].id if field.columns else None
        print(f"  field @id: {field.id} | name: {field.name} | column @id: {col_id} | dataType: {field.data_type}")
else:
    print("No record sets found!")

## 3. Data Extraction
Load records from the main record set into a DataFrame for analysis. All entity references use their `@id`.

In [ ]:
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for recordSet @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"Loaded shape: {df.shape}")
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Show sample columns and records from the main record set
if main_record_set_id and main_record_set_id in dataframes:
    print(f"\nDataFrame columns for '{main_record_set_id}':\n", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data filtering and normalization. All columns and fields are referenced strictly by `@id` (as shown above).

Here, we select example numeric and group fields using their `@id`, as obtained in the previous overview.

In [ ]:
# Select a numeric field and a group field for EDA
# First, print all columns for reference
print("\nColumns in main record set DataFrame:")
df = dataframes[main_record_set_id]
print(df.columns.tolist())

# Example selection based on common proteomics datasets--adjust as needed for your data
# (You may have e.g., 'cr:abundance', 'cr:molecular_weight', 'cr:peptide_count', etc. as columns)
numeric_field_id = None
group_field_id = None

# Try to auto-select a numeric field and a group field by heuristics
for col in df.columns:
    if numeric_field_id is None and any(key in col.lower() for key in ['abundance', 'amount', 'intensity', 'count', 'weight', 'coverage']):
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
    if group_field_id is None and any(key in col.lower() for key in ['group', 'sample', 'condition', 'type']):
        group_field_id = col

# Fallbacks if not found
if numeric_field_id is None:
    # just pick any numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
if group_field_id is None:
    group_field_id = df.columns[0]

print(f"\nSelected numeric field (@id): {numeric_field_id}")
print(f"Selected group field (@id): {group_field_id}")

threshold = df[numeric_field_id].mean() if numeric_field_id else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field_id} (showing means):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions and feature relationships. Here, we plot the distribution of the selected numeric field and group means (if available).

All data references are by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(10, 5))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Grouped barplot if group_field exists
if group_field_id and numeric_field_id and group_field_id in df.columns:
    plt.figure(figsize=(12, 6))
    order = df[group_field_id].value_counts().index[:10] # top 10
    sns.barplot(x=group_field_id, y=numeric_field_id, data=filtered_df, order=order)
    plt.title(f"Mean {numeric_field_id} by {group_field_id} (Top 10)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- We loaded and explored the dataset _Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells_ using Croissant schemas via `mlcroissant`.
- All entities and data elements were referenced using their stable `@id` fields for robust, schema-driven analysis.
- Data was filtered and normalized based on the selected numeric field, and we visualized the primary distributions and groupwise means.
- You can further analyze relationships, perform advanced filtering, or export selected record sets with confidence in the dataset's metadata provenance and structure thanks to Croissant.